# HerWILL Datathon: fitting the training data as closely as possible (HateBERT, Twitter-XLM-R and XLM-R Large)

**Task** (see `context.md`): classify Bangla / English / Banglish social media text into
`0 = Explicitly Toxic/Hateful`, `1 = Subtly Toxic`, `2 = Non-toxic/Neutral`.

The goal here is to **fit the training set as closely as possible** instead of generalizing, so
the usual regularization is switched off on purpose:

- **Dropout = 0 everywhere**: hidden, attention and classifier dropout are set to 0 in the config,
  and after loading, every `nn.Dropout` module is also forced to `p=0`.
- **Weight decay = 0**: plain AdamW with `weight_decay=0` for every parameter group.
- **Larger classification head**: `[CLS] ⊕ mean-pool` → MLP `2H → 4096 → 2048 → 1024 → 3`
  (GELU + LayerNorm, no dropout). This replaces the usual single `Linear(H, 3)`.
- No label smoothing, no early stopping, no layer-wise LR decay. More epochs (default 6), with
  linear decay to 0 so the last epochs settle into a sharp fit of the training data.
- `MAX_LEN = 256`, so less of each text is cut off.
- Gradient clipping (1.0) stays on. It keeps the runs stable and does not act as a regularizer.

| tag | checkpoint | params (encoder) | notes |
|---|---|---|---|
| `hatebert` | `GroNLP/hatebert` | ~110M | English-only BERT vocab; Bangla gets split into characters |
| `twitter_xlmr` | `cardiffnlp/twitter-xlm-roberta-base` | ~278M | multilingual, trained on tweets |
| `xlmr_large` | `FacebookAI/xlm-roberta-large` | ~560M | largest; lower LR |

**Data**: pass `train.csv` and `validation.csv` in explicitly (section 2, `TRAIN_PATH` / `VAL_PATH`).
Both use the columns `id, text, y`. If `validation.csv` has no label column, the notebook only writes
predictions for it. After every epoch we record **train loss**, **train acc / macro-F1**
(an eval-mode pass over the full train set, which is the memorization metric) and **validation
acc / macro-F1**.

**Runtime**: Kaggle **RTX PRO 6000 (96 GB)**, bf16 autocast, TF32 matmuls. Turn on **GPU**.

**Offline (Internet OFF)**: attach the `herwill-offline-assets` dataset (from
`00_download_offline_assets.ipynb`) plus your data dataset. The base encoders are loaded from
`<assets>/models/<folder>` (auto-found under `/kaggle/input`, or set `OFFLINE_ASSETS` in section 2).
The BanglaBERT normalizer installs from `<assets>/wheels`. A model that is missing from the assets falls
back to the Hub id, which needs Internet.

## 1. Config

In [ ]:
# (tag, hub id, folder name under <offline-assets>/models/)
MODELS = [
    ("hatebert",     "GroNLP/hatebert",                     "hatebert"),
    ("twitter_xlmr", "cardiffnlp/twitter-xlm-roberta-base", "twitter-xlm-roberta-base"),
    ("xlmr_large",   "FacebookAI/xlm-roberta-large",        "xlm-roberta-large"),
]

# Per-model overrides of CFG fields (restored after each model).
OVERRIDES = {
    "xlmr_large": {"LR": 1e-5, "WARMUP_RATIO": 0.1},
}


class CFG:
    # --- data ---
    MAX_LEN      = 256
    SEED         = 42

    # --- optimisation: no regularization on purpose ---
    EPOCHS       = 6
    LR           = 3e-5        # encoder LR (large models override to 1e-5)
    HEAD_LR      = 3e-4        # big MLP head is trained from scratch
    WEIGHT_DECAY = 0.0         # no weight decay
    DROPOUT      = 0.0         # no dropout anywhere
    LABEL_SMOOTHING = 0.0
    WARMUP_RATIO = 0.06
    TRAIN_BS     = 32
    EVAL_BS      = 256
    MAX_GRAD_NORM = 1.0        # stability only, not a regularizer
    BF16         = True

    # --- classification head ---
    HEAD_HIDDEN  = [4096, 2048, 1024]   # MLP widths on top of [CLS] ⊕ mean-pool (2H)

    # --- monitoring ---
    EVAL_TRAIN_EVERY = 1       # full eval pass over train every N epochs (memorization metric)

    DEBUG        = False       # True -> tiny subsample, 1 epoch, 1 model (smoke test)


N_CLASSES = 3
CLASS_NAMES = {0: "Explicitly Toxic", 1: "Subtly Toxic", 2: "Non-toxic"}

if CFG.DEBUG:
    CFG.EPOCHS = 1
    MODELS = MODELS[:1]

## 2. Environment, paths & imports

In [ ]:
import os, gc, re, json, time, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ---- set these to your uploaded files ----
TRAIN_PATH = Path("/kaggle/input/datasets/supriopaul844/uap-train/english.csv")   # e.g. Path("/kaggle/input/herwill-data/train.csv")
VAL_PATH   = Path("/kaggle/input/datasets/supriopaul844/uap-val/val_english.csv")   # e.g. Path("/kaggle/input/herwill-data/validation.csv")


def find_file(explicit, name):
    if explicit is not None:
        p = Path(explicit)
        if not p.exists():
            raise FileNotFoundError(p)
        return p
    hits = sorted(Path("/kaggle/input").rglob(name)) if Path("/kaggle/input").exists() else []
    hits += [p for p in (Path(name), Path("data") / name) if p.exists()]
    if not hits:
        raise FileNotFoundError(f"{name} not found; set it explicitly at the top of this cell.")
    if len(hits) > 1:
        print(f"multiple {name} found, using {hits[0]}:", *hits, sep="\n  ")
    return hits[0]


TRAIN_PATH = find_file(TRAIN_PATH, "train.csv")
VAL_PATH   = find_file(VAL_PATH, "validation.csv")

# ---- base encoders from the offline-assets dataset (00_download_offline_assets.ipynb) ----
# Layout: <dataset>/models/<local_dir>/{config.json, tokenizer files, model.safetensors}
OFFLINE_ASSETS = None   # e.g. Path("/kaggle/input/herwill-offline-assets"); auto-searched if None


def locate_offline_assets():
    if OFFLINE_ASSETS is not None:
        return Path(OFFLINE_ASSETS)
    if not Path("/kaggle/input").exists():
        return None
    wanted = {local for _, _, local in MODELS}
    for cfg in sorted(Path("/kaggle/input").rglob("models/*/config.json")):
        if cfg.parent.name in wanted:
            return cfg.parent.parent.parent
    return None


ASSETS = locate_offline_assets()
MODEL_PATHS = {}
for tag, hub_id, local in MODELS:
    p = ASSETS / "models" / local if ASSETS is not None else None
    if p is not None and (p / "config.json").exists():
        MODEL_PATHS[tag] = str(p)
    else:
        MODEL_PATHS[tag] = hub_id
        print(f"WARNING: {local} not found in offline assets -> will download {hub_id} (needs Internet)")

if all(Path(m).exists() for m in MODEL_PATHS.values()):
    os.environ["HF_HUB_OFFLINE"] = "1"          # set before transformers is imported
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("offline assets:", ASSETS)
for tag, path in MODEL_PATHS.items():
    print(f"  {tag:<18} {path}")

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
USE_BF16 = CFG.BF16 and DEVICE == "cuda" and torch.cuda.is_bf16_supported()

print("train     :", TRAIN_PATH)
print("validation:", VAL_PATH)
print("out dir   :", OUT_DIR)
print("device    :", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("bf16      :", USE_BF16)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(CFG.SEED)

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
import transformers
print("transformers", transformers.__version__)

## 3. Load data

In [ ]:
def load_csv(path, need_labels):
    df = pd.read_csv(path)
    print(f"{path.name}: {df.shape}  columns={list(df.columns)}")
    text_col = "text" if "text" in df.columns else next(
        c for c in df.columns if c.lower() in ("text", "sentence", "comment", "content"))
    label_col = "y" if "y" in df.columns else next(
        (c for c in df.columns if c.lower() in ("y", "label", "labels", "target", "class")), None)
    if need_labels and label_col is None:
        raise ValueError(f"{path} has no label column")
    id_col = "id" if "id" in df.columns else None

    out = pd.DataFrame({
        "id": df[id_col] if id_col else np.arange(len(df)),
        "text": df[text_col].fillna("").astype(str),
    })
    if label_col is not None:
        out["y"] = df[label_col]
        if need_labels:
            out = out.dropna(subset=["y"])
        out["y"] = out["y"].astype(int)
    return out.reset_index(drop=True)


train_raw = load_csv(TRAIN_PATH, need_labels=True)
val_raw   = load_csv(VAL_PATH, need_labels=False)
VAL_HAS_LABELS = "y" in val_raw.columns

if CFG.DEBUG:
    train_raw = train_raw.sample(400, random_state=CFG.SEED).reset_index(drop=True)
    val_raw   = val_raw.sample(min(200, len(val_raw)), random_state=CFG.SEED).reset_index(drop=True)

print("\ntrain rows:", len(train_raw), "  validation rows:", len(val_raw),
      "  validation labelled:", VAL_HAS_LABELS)
print(train_raw["y"].value_counts().sort_index().rename(CLASS_NAMES))
if VAL_HAS_LABELS:
    print(val_raw["y"].value_counts().sort_index().rename(CLASS_NAMES))

## 4. Text preprocessing

Same light cleaning as the earlier notebooks: NFC, strip zero-width characters, `[USER]` / `[URL]` tokens, squeeze elongation and repeated punctuation. No lowercasing, stopword removal or emoji stripping.

In [ ]:
import unicodedata as ud

ZW_RE      = re.compile(r"[\u200b\u200e\u200f\ufeff\u00ad\u2060]")
URL_RE     = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"(?<!\w)@\w+")
ELONG_RE   = re.compile(r"([A-Za-z])\1{2,}")
PUNCT_RE   = re.compile(r"([!?.,\-*])\1{2,}")
MAX_CHARS  = 2000


def clean_text(s):
    s = str(s)
    s = ud.normalize("NFC", s)
    s = ZW_RE.sub("", s)
    s = URL_RE.sub(" [URL] ", s)
    s = MENTION_RE.sub(" [USER] ", s)
    s = ELONG_RE.sub(r"\1\1", s)
    s = PUNCT_RE.sub(r"\1\1", s)
    s = s[:MAX_CHARS]
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else "[EMPTY]"


train_df = train_raw.assign(text=train_raw["text"].map(clean_text))
val_df   = val_raw.assign(text=val_raw["text"].map(clean_text))
print(f"train rows altered by cleaning: {(train_raw['text'] != train_df['text']).mean():.2%}")
print(train_df.head(5).to_string(index=False))

## 5. Tokenization & dynamic padding

Each model's tokenizer runs over the whole text list once. Eval loaders go through rows sorted by length, which keeps padding small; predictions are put back in the original order.

In [ ]:
class EncodedDS(Dataset):
    def __init__(self, enc, labels=None, order=None):
        self.enc = enc
        self.labels = None if labels is None else np.asarray(labels)
        self.order = np.arange(len(enc["input_ids"])) if order is None else np.asarray(order)

    def __len__(self):
        return len(self.order)

    def __getitem__(self, i):
        j = self.order[i]
        item = {k: v[j] for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = int(self.labels[j])
        return item


def make_collate(tok):
    def collate(batch):
        labels = [b.pop("labels") for b in batch] if "labels" in batch[0] else None
        out = tok.pad(batch, return_tensors="pt")
        if labels is not None:
            out["labels"] = torch.tensor(labels, dtype=torch.long)
        return out
    return collate


def encode(tok, texts):
    return tok(list(texts), truncation=True, max_length=CFG.MAX_LEN)


def eval_loader(enc, labels, collate):
    order = np.argsort([len(x) for x in enc["input_ids"]])
    ds = EncodedDS(enc, labels, order)
    return DataLoader(ds, batch_size=CFG.EVAL_BS, shuffle=False, collate_fn=collate,
                      num_workers=2, pin_memory=True), order

## 6. Model: encoder + large MLP head, no dropout

`AutoModel` loads the bare encoder. Every dropout probability in its config is set to 0, and after
loading, every remaining `nn.Dropout` module is forced to `p=0` too, which also catches ELECTRA/RoBERTa
submodules that read other config keys. The head takes `[CLS] ⊕ masked-mean-pool` (2H features)
and passes it through `CFG.HEAD_HIDDEN = [4096, 2048, 1024]` with GELU + LayerNorm, ending in a 3-way linear layer.

In [ ]:
DROPOUT_KEYS = ["hidden_dropout_prob", "attention_probs_dropout_prob", "classifier_dropout",
                "attention_dropout", "activation_dropout", "dropout", "summary_last_dropout",
                "pooler_dropout"]


class BigHeadClassifier(nn.Module):
    def __init__(self, model_name, n_classes, hidden_sizes):
        super().__init__()
        config = AutoConfig.from_pretrained(model_name)
        for k in DROPOUT_KEYS:
            if hasattr(config, k):
                setattr(config, k, CFG.DROPOUT)
        # BERT / RoBERTa have an unused pooler we drop; ELECTRA has none (and rejects the kwarg)
        extra = {"add_pooling_layer": False} if config.model_type in ("bert", "roberta", "xlm-roberta") else {}
        self.encoder = AutoModel.from_pretrained(model_name, config=config, **extra)
        H = config.hidden_size

        layers, d_in = [], 2 * H
        for d_out in hidden_sizes:
            layers += [nn.Linear(d_in, d_out), nn.LayerNorm(d_out), nn.GELU()]
            d_in = d_out
        layers.append(nn.Linear(d_in, n_classes))
        self.head = nn.Sequential(*layers)

        # belt and braces: kill every dropout module that might still exist
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.p = CFG.DROPOUT

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        h = self.encoder(**kw).last_hidden_state                    # (B, T, H)
        cls = h[:, 0]
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        mean = (h * m).sum(1) / m.sum(1).clamp(min=1.0)
        return self.head(torch.cat([cls, mean], dim=-1).float())


def count_params(module):
    return sum(p.numel() for p in module.parameters())


def build_optimizer(model):
    # weight decay is 0 for every group; only the LR differs between encoder and head
    return torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": CFG.LR},
        {"params": model.head.parameters(),    "lr": CFG.HEAD_LR},
    ], lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY, eps=1e-6)

## 7. Train / evaluate one model

Per epoch: train loss (running), then an eval-mode pass over **train** (memorization) and **validation**. Saves **one checkpoint per model, after the last epoch** (`{tag}_final/`). Its validation probabilities are written to disk so they can be ensembled later.

In [ ]:
@torch.no_grad()
def predict(model, loader, order):
    model.eval()
    chunks = []
    for batch in loader:
        batch.pop("labels", None)
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
            logits = model(**batch)
        chunks.append(torch.softmax(logits.float(), -1).cpu().numpy())
    probs_sorted = np.concatenate(chunks)
    probs = np.empty_like(probs_sorted)
    probs[order] = probs_sorted            # undo length sort
    return probs


def scores(probs, labels):
    pred = probs.argmax(1)
    nll = -np.log(np.clip(probs[np.arange(len(labels)), labels], 1e-12, None)).mean()
    return {"loss": float(nll), "acc": accuracy_score(labels, pred),
            "f1_macro": f1_score(labels, pred, average="macro"), "pred": pred}


def save_model(model, tok, model_name, tag, kind):
    d = OUT_DIR / f"{tag}_{kind}"
    d.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), d / "model_state.pt")
    tok.save_pretrained(d)
    model.encoder.config.save_pretrained(d)
    with open(d / "head_config.json", "w") as fh:
        json.dump({"model_name": model_name, "n_classes": N_CLASSES,
                   "hidden_sizes": CFG.HEAD_HIDDEN, "max_len": CFG.MAX_LEN}, fh, indent=2)
    print(f"  saved {kind} -> {d}")


def save_val_preds(probs, tag, kind):
    np.save(OUT_DIR / f"val_probs_{tag}_{kind}.npy", probs)
    pd.DataFrame({"id": val_df["id"], "y": probs.argmax(1),
                  **{f"p{i}": probs[:, i] for i in range(N_CLASSES)}}
                 ).to_csv(OUT_DIR / f"val_pred_{tag}_{kind}.csv", index=False)


def train_model(tag, model_name):
    saved = {}
    for k, val in OVERRIDES.get(tag, {}).items():
        saved[k] = getattr(CFG, k); setattr(CFG, k, val)
    print(f"\n{'#'*70}\n# {tag}  ({model_name})  overrides={OVERRIDES.get(tag, {})}\n{'#'*70}")
    set_seed(CFG.SEED)

    try:
        tok = AutoTokenizer.from_pretrained(model_name)
        collate = make_collate(tok)
        tr_enc = encode(tok, train_df.text)
        va_enc = encode(tok, val_df.text)
        lens = np.array([len(x) for x in tr_enc["input_ids"]])
        print(f"  train tokens: mean {lens.mean():.0f}  p95 {np.percentile(lens, 95):.0f}  "
              f"truncated at {CFG.MAX_LEN}: {(lens >= CFG.MAX_LEN).mean():.2%}")

        tr_labels = train_df.y.values
        va_labels = val_df.y.values if VAL_HAS_LABELS else None
        tr_loader = DataLoader(EncodedDS(tr_enc, tr_labels), batch_size=CFG.TRAIN_BS, shuffle=True,
                               collate_fn=collate, num_workers=2, pin_memory=True, drop_last=False)
        tr_eval_loader, tr_order = eval_loader(tr_enc, tr_labels, collate)
        va_loader, va_order = eval_loader(va_enc, va_labels, collate)

        model = BigHeadClassifier(model_name, N_CLASSES, CFG.HEAD_HIDDEN).to(DEVICE)
        n_enc, n_head = count_params(model.encoder), count_params(model.head)
        n_drop = sum(1 for m in model.modules() if isinstance(m, nn.Dropout) and m.p > 0)
        print(f"  params: encoder {n_enc/1e6:.1f}M  head {n_head/1e6:.2f}M  "
              f"total {(n_enc+n_head)/1e6:.1f}M  | active dropout modules: {n_drop}")

        optimizer = build_optimizer(model)
        total_steps = len(tr_loader) * CFG.EPOCHS
        scheduler = get_linear_schedule_with_warmup(
            optimizer, int(total_steps * CFG.WARMUP_RATIO), total_steps)
        criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTHING)

        history = {k: [] for k in ["train_loss", "train_eval_loss", "train_acc", "train_f1_macro",
                                   "val_loss", "val_acc", "val_f1_macro"]}
        best_f1, val_probs = -1.0, None
        t_start = time.time()

        for epoch in range(CFG.EPOCHS):
            model.train()
            running, n_seen, t0 = 0.0, 0, time.time()
            for step, batch in enumerate(tr_loader):
                labels = batch.pop("labels").to(DEVICE, non_blocking=True)
                batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
                with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                    logits = model(**batch)
                loss = criterion(logits.float(), labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.MAX_GRAD_NORM)
                optimizer.step(); scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                running += loss.item(); n_seen += 1
                if step % 200 == 0:
                    print(f"  ep{epoch+1} step {step}/{len(tr_loader)}  loss {running/n_seen:.4f}")
            history["train_loss"].append(running / max(n_seen, 1))

            msg = f"  -> epoch {epoch+1}/{CFG.EPOCHS}  train_loss {history['train_loss'][-1]:.4f}"
            if (epoch + 1) % CFG.EVAL_TRAIN_EVERY == 0 or epoch + 1 == CFG.EPOCHS:
                tr = scores(predict(model, tr_eval_loader, tr_order), tr_labels)
                history["train_eval_loss"].append(tr["loss"])
                history["train_acc"].append(tr["acc"])
                history["train_f1_macro"].append(tr["f1_macro"])
                msg += f"  | TRAIN acc {tr['acc']:.4f} f1 {tr['f1_macro']:.4f} nll {tr['loss']:.4f}"
            else:
                for k in ["train_eval_loss", "train_acc", "train_f1_macro"]:
                    history[k].append(np.nan)

            val_probs = predict(model, va_loader, va_order)
            if VAL_HAS_LABELS:
                va = scores(val_probs, va_labels)
                history["val_loss"].append(va["loss"])
                history["val_acc"].append(va["acc"])
                history["val_f1_macro"].append(va["f1_macro"])
                msg += f"  | VAL acc {va['acc']:.4f} f1 {va['f1_macro']:.4f} nll {va['loss']:.4f}"
                best_f1 = max(best_f1, va["f1_macro"])   # reported only, not checkpointed
            print(msg + f"  ({time.time()-t0:.0f}s)")

        minutes = (time.time() - t_start) / 60
        save_model(model, tok, model_name, tag, "final")
        save_val_preds(val_probs, tag, "final")

        print(f"[done] {tag}  final TRAIN acc {history['train_acc'][-1]:.4f}  "
              f"best VAL f1 {best_f1:.4f}  ({minutes:.1f} min)")

        final_val = scores(val_probs, va_labels) if VAL_HAS_LABELS else None
        if final_val is not None:
            print(f"\n### {tag}: final-epoch validation report")
            print(classification_report(va_labels, final_val["pred"], digits=3,
                                        target_names=[CLASS_NAMES[i] for i in range(N_CLASSES)]))

        result = {"tag": tag, "model_name": model_name, "history": history,
                  "best_val_f1_macro": best_f1, "final_val": final_val, "minutes": minutes,
                  "params": n_enc + n_head}
        with open(OUT_DIR / f"history_{tag}.json", "w") as fh:
            json.dump({**history, "best_val_f1_macro": best_f1, "minutes": minutes,
                       "params": n_enc + n_head}, fh, indent=2, default=float)
        return result
    finally:
        for k, val in saved.items():
            setattr(CFG, k, val)

## 8. Train all three backbones

If one backbone fails (for example with an OOM), the error is caught and the next one still runs.

In [ ]:
RESULTS = {}
for tag, _, _ in MODELS:
    try:
        RESULTS[tag] = train_model(tag, MODEL_PATHS[tag])
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"[FAILED] {tag}: {type(e).__name__}: {e}")
    gc.collect(); torch.cuda.empty_cache()

print("\ntrained backbones:", list(RESULTS))

## 9. Curves: memorization (train) vs validation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for tag, res in RESULTS.items():
    h = res["history"]
    ep = range(1, len(h["train_loss"]) + 1)
    axes[0].plot(ep, h["train_loss"], marker="o", label=f"{tag} train (running)")
    if h["val_loss"]:
        axes[0].plot(ep, h["val_loss"], marker="s", linestyle="--", label=f"{tag} val")
    axes[1].plot(ep, h["train_acc"], marker="o", label=f"{tag} train")
    if h["val_acc"]:
        axes[1].plot(ep, h["val_acc"], marker="s", linestyle="--", label=f"{tag} val")
    axes[2].plot(ep, h["train_f1_macro"], marker="o", label=f"{tag} train")
    if h["val_f1_macro"]:
        axes[2].plot(ep, h["val_f1_macro"], marker="s", linestyle="--", label=f"{tag} val")

for ax, t, yl in zip(axes, ["Loss", "Accuracy", "Macro-F1"], ["loss", "accuracy", "macro F1"]):
    ax.set_title(f"{t} per epoch (solid = train, dashed = val)")
    ax.set_xlabel("epoch"); ax.set_ylabel(yl); ax.grid(alpha=0.3); ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(OUT_DIR / "training_curves.png", dpi=150)
plt.show()

## 10. Confusion matrices (final epoch, validation)

In [ ]:
if VAL_HAS_LABELS and RESULTS:
    n = len(RESULTS)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    axes = np.atleast_1d(axes)
    for ax, (tag, res) in zip(axes, RESULTS.items()):
        ev = res["final_val"]
        cm = confusion_matrix(val_df.y.values, ev["pred"], labels=range(N_CLASSES))
        ax.imshow(cm, cmap="Blues")
        ax.set_title(f"{tag}\nval acc={ev['acc']:.3f}  f1_macro={ev['f1_macro']:.3f}")
        ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
        ax.set_xticklabels([CLASS_NAMES[i] for i in range(N_CLASSES)], rotation=30, ha="right")
        ax.set_yticklabels([CLASS_NAMES[i] for i in range(N_CLASSES)])
        ax.set_xlabel("predicted"); ax.set_ylabel("true")
        for i in range(N_CLASSES):
            for j in range(N_CLASSES):
                ax.text(j, i, cm[i, j], ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "confusion_matrices.png", dpi=150)
    plt.show()
else:
    print("validation has no labels; skipping confusion matrices")

## 11. Cross-model comparison

In [ ]:
summary = pd.DataFrame([
    {
        "model": tag,
        "params_M": round(res["params"] / 1e6, 1),
        "final_train_acc": res["history"]["train_acc"][-1],
        "final_train_f1": res["history"]["train_f1_macro"][-1],
        "final_val_acc": res["final_val"]["acc"] if res["final_val"] else np.nan,
        "final_val_f1": res["final_val"]["f1_macro"] if res["final_val"] else np.nan,
        "best_val_f1": res["best_val_f1_macro"] if res["best_val_f1_macro"] >= 0 else np.nan,
        "minutes": round(res["minutes"], 1),
    }
    for tag, res in RESULTS.items()
])
print(summary.to_string(index=False))
summary.to_csv(OUT_DIR / "model_comparison.csv", index=False)

if len(summary):
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(summary)); w = 0.2
    ax.bar(x - 1.5*w, summary["final_train_acc"], w, label="train acc")
    ax.bar(x - 0.5*w, summary["final_train_f1"], w, label="train macro-F1")
    ax.bar(x + 0.5*w, summary["final_val_acc"], w, label="val acc")
    ax.bar(x + 1.5*w, summary["final_val_f1"], w, label="val macro-F1")
    ax.set_xticks(x); ax.set_xticklabels(summary["model"], rotation=15, ha="right")
    ax.set_ylim(0, 1.02); ax.set_title("Final-epoch metrics: train (memorization) vs validation")
    ax.grid(axis="y", alpha=0.3); ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "model_comparison.png", dpi=150)
    plt.show()

## 12. Validation-probability ensemble (mean of final-epoch models)

In [ ]:
probs = [np.load(OUT_DIR / f"val_probs_{tag}_final.npy") for tag in RESULTS]
if probs:
    ens = np.mean(probs, axis=0)
    pd.DataFrame({"id": val_df["id"], "y": ens.argmax(1),
                  **{f"p{i}": ens[:, i] for i in range(N_CLASSES)}}
                 ).to_csv(OUT_DIR / "val_pred_ensemble_final.csv", index=False)
    if VAL_HAS_LABELS:
        e = scores(ens, val_df.y.values)
        print(f"ensemble (final) VAL acc {e['acc']:.4f}  macro-F1 {e['f1_macro']:.4f}")

## 13. Reloading a saved checkpoint

In [ ]:
def load_checkpoint(ckpt_dir):
    ckpt_dir = Path(ckpt_dir)
    meta = json.load(open(ckpt_dir / "head_config.json"))
    tok = AutoTokenizer.from_pretrained(ckpt_dir)
    model = BigHeadClassifier(meta["model_name"], meta["n_classes"], meta["hidden_sizes"])
    model.load_state_dict(torch.load(ckpt_dir / "model_state.pt", map_location="cpu"))
    return model.to(DEVICE).eval(), tok

print("Artifacts in", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    print("  ", p.name)